# w04_baseline_score.ipynb

## User Intent Lane — Baseline Score & Rule

This notebook checks signals, encodes a rule, and reviews the top 10 results.

## 1. Two Signal Checks

### Signal A: CTR vs Position Tier
**FlyRank Flag Link:** CTR-fix logic — pages with low CTR for their position tier

**Verdict:** [CONFIRMED / OPPOSITE / MIXED / FALSE]

In [ ]:
# Connect to the warehouse
import duckdb
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
SAMPLE = f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')"

print("✅ Connected to Hugging Face!")

In [ ]:
# SIGNAL A: CTR vs Position Tier
signal_a = con.sql(f"""
    SELECT 
        CASE 
            WHEN gsc_avg_position <= 3 THEN 'Top (1-3)'
            WHEN gsc_avg_position <= 10 THEN 'Middle (4-10)'
            ELSE 'Bottom (11+)'
        END AS position_tier,
        COUNT(DISTINCT content_hash_id) AS n,
        AVG(ctr) AS avg_ctr
    FROM (
        SELECT 
            content_hash_id,
            gsc_avg_position,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
        FROM {SAMPLE}
        WHERE report_date = '2026-06-01'
        GROUP BY content_hash_id, gsc_avg_position
        HAVING SUM(gsc_impressions) >= 10
    )
    GROUP BY position_tier
    ORDER BY 
        CASE 
            WHEN position_tier = 'Top (1-3)' THEN 1
            WHEN position_tier = 'Middle (4-10)' THEN 2
            ELSE 3
        END
""").df()

print("SIGNAL A: CTR vs Position Tier")
print(f"n = {signal_a['n'].sum()}")
display(signal_a)

# Verdict
print("\n📌 Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]")
print("Why: [One sentence explanation]")

### Signal B: Staleness
**FlyRank Flag Link:** Refresh flags — pages that haven't been updated recently

**Verdict:** [CONFIRMED / OPPOSITE / MIXED / FALSE]

In [ ]:
# SIGNAL B: Content Age vs Engagement
signal_b = con.sql(f"""
    SELECT 
        CASE 
            WHEN d.content_age_days <= 30 THEN 'New (0-30 days)'
            WHEN d.content_age_days <= 90 THEN 'Mid (31-90 days)'
            ELSE 'Old (91+ days)'
        END AS age_bucket,
        COUNT(DISTINCT f.content_hash_id) AS n,
        AVG(f.ga4_engaged_sessions) AS avg_engagement,
        AVG(f.gsc_impressions) AS avg_impressions
    FROM {SAMPLE} f
    JOIN read_parquet('{REL}/dim_content.parquet') d 
        ON f.content_hash_id = d.content_hash_id
    WHERE f.report_date = '2026-06-01'
    GROUP BY age_bucket
    ORDER BY 
        CASE 
            WHEN age_bucket = 'New (0-30 days)' THEN 1
            WHEN age_bucket = 'Mid (31-90 days)' THEN 2
            ELSE 3
        END
""").df()

print("SIGNAL B: Content Age vs Engagement")
print(f"n = {signal_b['n'].sum()}")
display(signal_b)

# Verdict
print("\n📌 Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]")
print("Why: [One sentence explanation]")

## 2. Encode One Rule

**Rule:** Pages with CTR below 0.02 AND avg_position above 10 need refresh

**Score:** CTR / avg_position (lower CTR + worse position = higher score)

**Reason Code:** `LOW_CTR_POOR_POSITION`

**Action Label:** `REFRESH`

In [ ]:
# Write the ranked queue
import os
import pandas as pd

os.makedirs('work/outputs', exist_ok=True)

# Pull data
queue_data = con.sql(f"""
    SELECT 
        content_hash_id,
        ANY_VALUE(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_impressions) AS impressions
    FROM {SAMPLE}
    WHERE report_date = '2026-06-01'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) >= 10
""").df()

# Score: lower CTR + higher position = higher score
queue_data['score'] = (1 / (queue_data['ctr'] + 0.001)) * (queue_data['avg_position'] / 10)
queue_data['reason_code'] = 'LOW_CTR_POOR_POSITION'
queue_data['action_label'] = 'REFRESH'

# Sort by score descending
queue_data = queue_data.sort_values('score', ascending=False)

# Save CSV
queue_data.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("✅ Ranked queue written to work/outputs/baseline_action_score.csv")
print(f"Total pages scored: {len(queue_data)}")
display(queue_data.head(10))

## 3. Top-10 Review

For each of the top 10 pages:

In [ ]:
# Review the top 10
print("TOP 10 REVIEW")
print("=" * 80)

for idx, row in queue_data.head(10).iterrows():
    print(f"\nRank {idx+1}: content_hash_id = {row['content_hash_id']}")
    print(f"  Action: {row['action_label']}")
    print(f"  Why: Position {row['avg_position']:.1f} | CTR {row['ctr']:.4f} | Score {row['score']:.2f}")
    print(f"  What would make it wrong: [One sentence]")
    print("-" * 40)

## 4. Self-Check

✅ I've checked two signals (CTR vs Position, Staleness)

✅ At least one signal is flag-linked (CTR vs Position)

✅ I've written one rule with a score, reason code, and action label

✅ I've written the ranked queue to `work/outputs/baseline_action_score.csv`

✅ I've reviewed 10 rows with 'what would make it wrong' for each

✅ I've used no future-window or label-derived inputs

**Limitation:** This rule assumes CTR and position are the best signals for identifying pages that need refresh. It may miss pages with engagement issues not captured by these metrics.